# 04 — Modelling

**Goal.** Train and compare models against simple baselines, pick the best, and save artifacts for the backend.

**Models:**
- **Baselines** — global mean, same-day-last-week, 7-day moving average. If our ML models can't beat these, they don't earn their complexity.
- **Random Forest** (log target) — ensemble of decision trees, good for non-linearity and interactions.
- **XGBoost** (log target) — gradient boosting, typically best on tabular time-series with engineered features.
- **Prophet** — decomposes into trend + seasonality + holidays, doesn't use our 87 engineered features.

**Split.** Last 90 days as hold-out test. Random shuffling leaks future info and inflates metrics.

In [ ]:
import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)

REPO = Path.cwd().parent
df = pd.read_csv(REPO / 'data' / 'processed' / 'clinic_patients_engineered.csv', parse_dates=['date'])
with open(REPO / 'ml-pipeline' / 'models' / 'feature_list.json') as f:
    features = json.load(f)['features']
print(f'{len(df)} rows, {len(features)} features')

## 1. Train/test split

In [ ]:
TEST_DAYS = 90
df = df.sort_values('date').reset_index(drop=True)
split_idx = len(df) - TEST_DAYS
train = df.iloc[:split_idx].copy()
test  = df.iloc[split_idx:].copy()

X_train = train[features]
X_test  = test[features]
y_train = train['patients']
y_test  = test['patients']
y_train_log = train['patients_log']

print(f'Train: {len(train)} rows  {train.date.min().date()} → {train.date.max().date()}')
print(f'Test : {len(test)} rows   {test.date.min().date()} → {test.date.max().date()}')

## 2. Metric helper

In [ ]:
def metrics(y_true, y_pred, name):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    bias = np.mean(y_pred - y_true)
    return {'Model': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2, 'MAPE%': mape, 'Bias': bias}

## 3. Baselines

In [ ]:
results = []
preds = {}

preds['Baseline (mean)'] = np.full(len(test), y_train.mean())
results.append(metrics(y_test, preds['Baseline (mean)'], 'Baseline (mean)'))

preds['Baseline (last week)'] = test['patients_lag_7'].values
results.append(metrics(y_test, preds['Baseline (last week)'], 'Baseline (last week)'))

preds['Baseline (7d MA)'] = test['rolling_mean_7d'].values
results.append(metrics(y_test, preds['Baseline (7d MA)'], 'Baseline (7d MA)'))

pd.DataFrame(results).round(2)

## 4. Random Forest (log target)

In [ ]:
rf = RandomForestRegressor(
    n_estimators=300,
    min_samples_leaf=5,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train, y_train_log)
preds['Random Forest'] = np.expm1(rf.predict(X_test))
results.append(metrics(y_test, preds['Random Forest'], 'Random Forest'))
print(metrics(y_test, preds['Random Forest'], 'Random Forest'))

## 5. XGBoost (log target)

In [ ]:
xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    verbosity=0,
)
xgb_model.fit(X_train, y_train_log)
preds['XGBoost'] = np.expm1(xgb_model.predict(X_test))
results.append(metrics(y_test, preds['XGBoost'], 'XGBoost'))
print(metrics(y_test, preds['XGBoost'], 'XGBoost'))

## 6. Prophet

Prophet doesn't use our engineered features — it decomposes the series itself and we feed a few external regressors.

In [ ]:
from prophet import Prophet
import logging
logging.getLogger('prophet').setLevel(logging.WARNING)
logging.getLogger('cmdstanpy').setLevel(logging.WARNING)

regressor_cols = ['is_public_holiday', 'is_day_after_public_holiday', 'is_school_holiday',
                  'is_flu_peak', 'is_hayfever_season', 'temperature', 'pollen_index']

prophet_train = train[['date', 'patients'] + regressor_cols].rename(columns={'date': 'ds', 'patients': 'y'})
prophet_test  = test[['date'] + regressor_cols].rename(columns={'date': 'ds'})

prop = Prophet(
    yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False,
    seasonality_mode='multiplicative',
)
for c in regressor_cols:
    prop.add_regressor(c)
prop.fit(prophet_train)

forecast = prop.predict(prophet_test)
preds['Prophet'] = forecast['yhat'].values
results.append(metrics(y_test, preds['Prophet'], 'Prophet'))
print(metrics(y_test, preds['Prophet'], 'Prophet'))

## 7. Results table

In [ ]:
results_df = pd.DataFrame(results).sort_values('MAE').reset_index(drop=True)
display_df = results_df.copy()
for c in ['MAE', 'RMSE', 'MAPE%', 'Bias']:
    display_df[c] = display_df[c].round(1)
display_df['R2'] = display_df['R2'].round(3)
display_df

## 8. Actual vs predicted

In [ ]:
fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(test['date'], y_test.values, color='black', linewidth=2, label='Actual')
ax.plot(test['date'], preds['XGBoost'], color='#c0392b', linewidth=1.6, alpha=0.9, label='XGBoost')
ax.plot(test['date'], preds['Random Forest'], color='#3498db', linewidth=1.2, alpha=0.7, label='Random Forest')
ax.plot(test['date'], preds['Prophet'], color='#27ae60', linewidth=1.2, alpha=0.7, label='Prophet')
ax.set_title('Actual vs predicted — 90-day test window')
ax.set_ylabel('Patients')
ax.legend()
plt.tight_layout(); plt.show()

## 9. CV residuals + save artifacts

In-sample residuals are useless for uncertainty intervals — a gradient-boosted model with 500 trees on 87 features will memorise training data and return residuals close to zero. That gives absurdly narrow prediction intervals.

Instead, we use **TimeSeriesSplit** to get out-of-fold residuals on the training data. Each fold trains on past data only and predicts a held-out forward window — the residuals from those folds reflect realistic prediction error on unseen data, which is what we want for honest bootstrap intervals.

`features.json` is the contract the backend reads — ordered feature list, best-model name, and CV-residual stats for prediction intervals.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

models_dir = REPO / 'ml-pipeline' / 'models'
models_dir.mkdir(parents=True, exist_ok=True)

# --- TimeSeriesSplit out-of-fold residuals for honest prediction intervals ---
tscv = TimeSeriesSplit(n_splits=5, test_size=60)
cv_residuals = []
for tr_idx, va_idx in tscv.split(X_train):
    fold_model = xgb.XGBRegressor(**xgb_model.get_params())
    fold_model.fit(X_train.iloc[tr_idx], y_train_log.iloc[tr_idx])
    fold_pred = np.expm1(fold_model.predict(X_train.iloc[va_idx]))
    fold_actual = y_train.iloc[va_idx].values
    cv_residuals.extend(fold_actual - fold_pred)
cv_residuals = np.asarray(cv_residuals)
print(f'CV residuals: n={len(cv_residuals)}, mean={cv_residuals.mean():.2f}, std={cv_residuals.std():.2f}')

# --- Save model artifacts ---
xgb_model.save_model(models_dir / 'xgboost.json')
joblib.dump(rf, models_dir / 'random_forest.joblib')

best_model_name = results_df.iloc[0]['Model'].lower().replace(' ', '_')

features_json = {
    'features': features,
    'n_features': len(features),
    'best_model': best_model_name,
    'target_transform': 'log1p' if best_model_name in ('xgboost', 'random_forest') else 'none',
    'cv_residual_std': float(np.std(cv_residuals)),
    'cv_residual_mean': float(np.mean(cv_residuals)),
    'test_period': {'start': str(test.date.min().date()), 'end': str(test.date.max().date())},
    'tiers': {'low_max': 80, 'medium_max': 150},
}
with open(models_dir / 'features.json', 'w') as f:
    json.dump(features_json, f, indent=2)

results_df.to_csv(models_dir / 'model_results.csv', index=False)

test_out = test[['date', 'patients']].copy()
for name, p in preds.items():
    test_out[f'pred_{name}'] = p
test_out.to_csv(models_dir / 'test_predictions.csv', index=False)

np.save(models_dir / 'cv_residuals.npy', cv_residuals)

print(f'\nBest model: {best_model_name}')
print('Saved artifacts:')
for p in sorted(models_dir.iterdir()):
    print(f'  {p.name}')